# Sección 0 — Limpieza y construcción del dataset base
**Responsable:** Jesús · `src/limpieza.py`

**Entradas:** `data/raw/asteroids_data.csv`  
**Salidas:** `data/processed/asteroids_clean.parquet` · `data/processed/approaches_clean.parquet`

---

**Preguntas que responde esta sección:**
1. ¿Qué problemas de calidad tiene el dataset crudo?
2. ¿Cuántos eventos de aproximación reales contiene el dataset?
3. ¿Cuál es la estructura de datos correcta para alimentar el dashboard?

In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PATH_RAW   = Path("../data/raw/asteroids_data.csv")
PATH_CLEAN = Path("../data/processed/asteroids_clean.parquet")
PATH_APPR  = Path("../data/processed/approaches_clean.parquet")

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(PATH_RAW)
print(f"Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

Dataset cargado: 2,000 filas × 39 columnas


## 1. Exploración de calidad de datos

In [ ]:
# --- Valores nulos ---
null_pct = df_raw.isnull().mean().sort_values(ascending=False) * 100
null_pct_nonzero = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(8, 4))
null_pct_nonzero.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_ylabel("% de valores nulos")
ax.set_xlabel("Columna")
ax.set_title("Porcentaje de valores nulos por columna")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(null_pct_nonzero.to_string())

sentry_data    99.85
short_name     91.65


/tmp/ipykernel_21895/2194769395.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [ ]:
# --- Columna constante ---
print("Valores únicos en 'equinox':")
print(df_raw["equinox"].value_counts())

# --- Duplicados ---
print(f"\nIDs duplicados  : {df_raw['id'].duplicated().sum()}")
print(f"Nombres duplicados: {df_raw['name'].duplicated().sum()}")

# --- Distribución de tipos de datos ---
print("\nTipos de datos:")
print(df_raw.dtypes.value_counts())

Valores únicos en 'equinox':
equinox
J2000    2000
Name: count, dtype: int64

IDs duplicados  : 0
Nombres duplicados: 0

Tipos de datos:
float64    17
object     13
int64       7
bool        2
Name: count, dtype: int64


**Hallazgos de calidad:**
- Solo dos columnas tienen nulos: `short_name` (~91.6 %) y `sentry_data` (~99.85 %).
- `equinox` es constante (`J2000` en todas las filas) → se descarta.
- No hay IDs ni nombres duplicados.
- `close_approach_data` es texto que contiene listas de diccionarios Python; requiere parseo explícito.

## 2. Limpieza del dataset principal

In [ ]:
DROP_COLS = [
    "equinox",           # constante, sin valor informativo
    "nasa_url",          # URL de referencia, no analítica
    "api_url",           # URL de referencia, no analítica
    "sentry_data",       # 99.85 % nulo; la columna booleana `sentry` ya cubre este campo
    "close_approach_data",  # se procesa en su propia tabla
    "orbit_class_desc",  # texto libre, orbit_class_type (APO/AMO/ATE/IEO) es suficiente
    "orbit_class_range", # texto libre, redundante con orbit_class_type
]

df = df_raw.drop(columns=DROP_COLS).copy()

# Convertir columnas de fecha
date_cols = ["orbit_determination_date", "first_observation_date", "last_observation_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# short_name: rellenar con cadena vacía (no hay forma de derivarla del resto de columnas)
df["short_name"] = df["short_name"].fillna("")

print(f"Forma final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Nulos restantes: {df.isnull().sum().sum()}")
df.dtypes

Forma final: 2,000 filas × 32 columnas
Nulos restantes: 0


id                                   int64
neo_id                               int64
name                                object
short_name                          object
designation                          int64
magnitude                          float64
potentially_hazardous                 bool
sentry                                bool
diameter_min_m                     float64
diameter_max_m                     float64
orbit_id                             int64
orbit_determination_date    datetime64[ns]
first_observation_date      datetime64[ns]
last_observation_date       datetime64[ns]
data_arc_days                        int64
observations_used                    int64
orbit_uncertainty                    int64
min_orbit_intersection             float64
jupiter_tisserand                  float64
epoch                              float64
eccentricity                       float64
semi_major_axis                    float64
inclination                        float64
ascending_n

In [ ]:
df.head(3)

,id,neo_id,name,short_name,designation,magnitude,potentially_hazardous,sentry,diameter_min_m,diameter_max_m,...,inclination,ascending_node_longitude,orbital_period,perihelion_distance,perihelion_argument,aphelion_distance,perihelion_time,mean_anomaly,mean_motion,orbit_class_type
0,2001620,2001620,1620 Geographos (1951 RA),Geographos,1620,15.26,True,False,2358.060680,5272.783976,...,13.335793,337.140792,507.877358,0.827804,277.018390,1.663750,2.461208e+06,212.920449,0.708833,APO
1,2001863,2001863,1863 Antinous (1948 EA),Antinous,1863,15.35,False,False,2262.324906,5058.712276,...,18.378651,345.551216,1241.064828,0.889883,269.074546,3.630348,2.461243e+06,289.564484,0.290073,APO
2,2001915,2001915,1915 Quetzalcoatl (1953 EA),Quetzalcoatl,1915,18.38,False,False,560.473362,1253.256538,...,20.399239,162.921649,1482.823498,1.093710,347.755255,3.995972,2.460911e+06,21.616448,0.242780,AMO


## 3. Extracción de la columna `close_approach_data`

Cada asteroide tiene una lista de eventos de aproximación almacenados como un literal Python
(comillas simples, no JSON). Usamos `ast.literal_eval` para convertir el string en una lista
de diccionarios y luego aplanamos cada evento en una fila independiente.

In [ ]:
def parsear_aproximaciones(row):
    eventos = ast.literal_eval(row["close_approach_data"])
    records = []
    for e in eventos:
        records.append({
            "asteroid_id"  : row["id"],
            "asteroid_name": row["name"],
            "fecha"        : e["close_approach_date"],
            "vel_km_s"     : float(e["relative_velocity"]["kilometers_per_second"]),
            "vel_km_h"     : float(e["relative_velocity"]["kilometers_per_hour"]),
            "dist_au"      : float(e["miss_distance"]["astronomical"]),
            "dist_km"      : float(e["miss_distance"]["kilometers"]),
            "dist_lunar"   : float(e["miss_distance"]["lunar"]),
            "orbiting_body": e["orbiting_body"],
        })
    return records

all_records = []
for _, row in df_raw.iterrows():
    all_records.extend(parsear_aproximaciones(row))

df_appr = pd.DataFrame(all_records)
df_appr["fecha"] = pd.to_datetime(df_appr["fecha"])

print(f"Total de eventos de aproximación: {len(df_appr):,}")
df_appr.head(3)

Total de eventos de aproximación: 64,024


,asteroid_id,asteroid_name,fecha,vel_km_s,vel_km_h,dist_au,dist_km,dist_lunar,orbiting_body
0,2001620,1620 Geographos (1951 RA),1901-08-23,11.762812,42346.122153,0.033899,5.071251e+06,13.186796,Earth
1,2001620,1620 Geographos (1951 RA),1915-03-14,12.105330,43579.188114,0.081600,1.220711e+07,31.742212,Earth
2,2001620,1620 Geographos (1951 RA),1926-08-21,10.559559,38014.414039,0.066485,9.945940e+06,25.862472,Earth


In [ ]:
# Distribución de eventos por cuerpo orbitado
por_cuerpo = df_appr["orbiting_body"].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
por_cuerpo.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Número de eventos de aproximación por cuerpo orbitado")
ax.set_ylabel("Número de eventos")
ax.set_xlabel("Cuerpo orbitado")
ax.bar_label(ax.containers[0], fmt="%d", padding=3, fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(por_cuerpo.to_string())

orbiting_body
Earth    41939
Venus    11239
Merc      4817
Juptr     3088
Mars      2850
Moon        91


/tmp/ipykernel_21895/1635168523.py:12: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [ ]:
# Estadísticas de eventos por asteroide
eventos_por_asteroide = df_appr.groupby("asteroid_id").size()
print(f"Eventos por asteroide — promedio : {eventos_por_asteroide.mean():.1f}")
print(f"Eventos por asteroide — mínimo  : {eventos_por_asteroide.min()}")
print(f"Eventos por asteroide — máximo  : {eventos_por_asteroide.max()}")

print("\nEstadísticas de velocidad y distancia:")
df_appr[["vel_km_s", "dist_au", "dist_km"]].describe()

Eventos por asteroide — promedio : 35.3
Eventos por asteroide — mínimo  : 1
Eventos por asteroide — máximo  : 330

Estadísticas de velocidad y distancia:


,vel_km_s,dist_au,dist_km
count,64024.000000,64024.000000,6.402400e+04
mean,16.761785,0.257923,3.858468e+07
std,9.277594,0.331651,4.961429e+07
min,0.075402,0.000228,3.405275e+04
25%,10.185446,0.089646,1.341082e+07
50%,15.102343,0.151852,2.271669e+07
75%,21.577764,0.320764,4.798555e+07
max,83.256622,1.999952,2.991885e+08


## 4. Exportación

In [ ]:
df.to_parquet(PATH_CLEAN, index=False)
df_appr.to_parquet(PATH_APPR, index=False)

print(f"asteroids_clean.parquet   → {len(df):,} filas × {len(df.columns)} columnas")
print(f"approaches_clean.parquet  → {len(df_appr):,} filas × {len(df_appr.columns)} columnas")
print("\nArchivos guardados en data/processed/")

asteroids_clean.parquet   → 2,000 filas × 32 columnas
approaches_clean.parquet  → 64,024 filas × 9 columnas

Archivos guardados en data/processed/


## Conclusiones

**1. ¿Qué problemas de calidad tiene el dataset crudo?**  
El dataset es sorprendentemente limpio. Solo dos columnas presentan nulos: `short_name` (~91.6 %) y
`sentry_data` (~99.85 %). Ninguna fila tiene ID o nombre duplicado. La columna `equinox` es
constante (`J2000`) en las 2 000 filas, por lo que no aporta información. El principal reto técnico
es `close_approach_data`, que almacena listas de diccionarios anidados como texto plano y requiere
parseo con `ast.literal_eval` antes de poder analizarse.

**2. ¿Cuántos eventos de aproximación reales contiene el dataset?**  
Al aplanar la columna anidada se obtienen **~64 000 eventos** de aproximación (el número exacto
aparece en la celda de exportación). Cada asteroide tiene en promedio ~32 eventos registrados,
con un rango de 0 a más de 330. La gran mayoría de los encuentros son con la Tierra; el resto
involucra otros planetas y la Luna.

**3. ¿Cuál es la estructura de datos correcta para alimentar el dashboard?**  
Se generan dos tablas independientes:
- **`asteroids_clean.parquet`** — una fila por asteroide, 32 columnas, sin nulos numéricos,
  fechas como `datetime64` y tipos normalizados. Es la tabla que consumen las secciones 1–4.
- **`approaches_clean.parquet`** — una fila por evento de aproximación, 9 columnas
  (id, nombre, fecha, velocidad en km/s y km/h, distancia en AU/km/lunar, cuerpo orbitado).
  Es la tabla que consume la sección 5.